# Indicadores de Muestras por Cuadrante - Manizales 2021-2024

## Análisis con IDBA Poisson-Gamma

**Objetivo:** Construir indicadores por cuadrante (2021-2024) con conteos, días de actividad, densidades naive e IDBA (Empirical Bayes robusto).

**Archivos de entrada:**
- `muestras_MANIZALES_2021-2024.csv`: Datos de muestras con columnas id, fecha_evento, id_autor, lat, lot, cod_cuadrante
- `manizales_metricas.csv`: Métricas geométricas con cod_cuadrante y area_m2

**Archivos de salida:**
- `indicadores_cuadrantes_MANIZALES_2021-2024.csv`: Dataset final con todos los indicadores
- `idba_priors_MANIZALES_2021-2024.csv`: Parámetros Bayesianos por año

**Modelo IDBA:** $k|ρ,A \sim \text{Poisson}(ρA)$, $ρ \sim \text{Gamma}(α,β)$ → $E[ρ|k,A] = \frac{α+k}{β+A}$

In [1]:
# ================================================================================================
# 1. IMPORTS Y CONFIGURACIÓN
# ================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 50)

# Suprimir warnings no críticos
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# Años de análisis
ANIOS = [2021, 2022, 2023, 2024]

# Escala para normalización (tasas por 1000 m²)  
SCALE = 1000

# Configuración de archivos (mismo directorio del notebook)
BASE_DIR = Path('.')
ARCHIVO_MUESTRAS = BASE_DIR / 'muestras_MANIZALES_2021-2024.csv'
ARCHIVO_METRICAS = BASE_DIR / 'manizales_metricas.csv'
ARCHIVO_SALIDA_INDICADORES = BASE_DIR / 'indicadores_cuadrantes_MANIZALES_2021-2024.csv'
ARCHIVO_SALIDA_PRIORS = BASE_DIR / 'idba_priors_MANIZALES_2021-2024.csv'

print("✅ Configuración completada")
print(f"📊 Años de análisis: {ANIOS}")
print(f"📏 Escala de normalización: {SCALE} (por 1000 m²)")
print(f"📁 Directorio base: {BASE_DIR.absolute()}")

✅ Configuración completada
📊 Años de análisis: [2021, 2022, 2023, 2024]
📏 Escala de normalización: 1000 (por 1000 m²)
📁 Directorio base: c:\Users\ESP_NEGOCIO\Documents\GitHub\MAPAS_TA_DEV_1\tools


In [5]:
# ================================================================================================
# 2. CARGA Y PREPARACIÓN DE DATOS
# ================================================================================================

print("📂 Cargando datos...")

# =============================================================================
# CARGA DE DATOS CON PARÁMETROS ESPECÍFICOS
# =============================================================================

# Cargar datos de muestras (sep=';', encoding='utf-8-sig')
print(f"   Leyendo muestras desde: {ARCHIVO_MUESTRAS}")
df_m = pd.read_csv(ARCHIVO_MUESTRAS, sep=';', encoding='utf-8-sig')

# Cargar métricas de cuadrantes (sep=';', encoding='utf-8-sig')  
print(f"   Leyendo métricas desde: {ARCHIVO_METRICAS}")
df_q = pd.read_csv(ARCHIVO_METRICAS, encoding='utf-8-sig')

print(f"📊 Muestras cargadas: {len(df_m):,} registros")
print(f"📐 Cuadrantes cargados: {len(df_q):,} cuadrantes")

print(df_m.columns)
print(df_q.columns)

# =============================================================================
# PREPARACIÓN DE FECHAS Y AÑOS
# =============================================================================

# Convertir fecha_evento a datetime
df_m['fecha_evento'] = pd.to_datetime(df_m['fecha_evento'], errors='coerce')

# Crear columna ANIO
df_m['ANIO'] = df_m['fecha_evento'].dt.year

# Filtrar solo años de interés y remover fechas inválidas
df_m_original = df_m.copy()
df_m = df_m[df_m['ANIO'].isin(ANIOS) & df_m['fecha_evento'].notna()]

print(f"📅 Registros válidos en período {min(ANIOS)}-{max(ANIOS)}: {len(df_m):,}")
print(f"🗑️ Registros filtrados: {len(df_m_original) - len(df_m):,}")

# =============================================================================
# NORMALIZACIÓN DE CÓDIGOS DE CUADRANTE
# =============================================================================

# Normalizar códigos en muestras: str → upper → strip → fillna('FUERA')
df_m['cod_cuadrante'] = (df_m['cod_cuadrante']
                        .astype(str)
                        .str.upper()
                        .str.strip()
                        .replace('NAN', np.nan)  # Convertir 'nan' string a NaN real
                        .fillna('FUERA'))

# Normalizar códigos en métricas (renombrar si necesario)
if 'codigo' in df_q.columns and 'cod_cuadrante' not in df_q.columns:
    df_q = df_q.rename(columns={'codigo': 'cod_cuadrante'})
    print("🔄 Renombrado 'codigo' → 'cod_cuadrante' en métricas")

df_q['cod_cuadrante'] = df_q['cod_cuadrante'].astype(str).str.upper().str.strip()

# =============================================================================
# PREPARACIÓN DE ÁREAS
# =============================================================================

# Validar y preparar columna de área
if 'area_m2' not in df_q.columns:
    if 'area' in df_q.columns:
        print("🔄 Convirtiendo 'area' → 'area_m2'")
        # Detectar si está en km² (valores típicamente < 100) vs m² (valores > 1000)
        sample_area = df_q['area'].dropna().iloc[:10] if len(df_q['area'].dropna()) > 0 else pd.Series([1])
        if sample_area.mean() < 100:  # Probablemente km²
            df_q['area_m2'] = df_q['area'] * 1e6  # km² → m²
            print("   Detectado área en km², convertido a m²")
        else:
            df_q['area_m2'] = df_q['area']  # Ya en m²
            print("   Área ya en m², copiado directamente")
    else:
        raise ValueError("No se encontró columna de área (area_m2 o area)")

# Crear area_km2 para legibilidad (solo informativa)
df_q['area_km2'] = df_q['area_m2'] / 1e6

print(f"📏 Área promedio: {df_q['area_m2'].mean():,.0f} m² ({df_q['area_km2'].mean():.3f} km²)")
print(f"📏 Área mediana: {df_q['area_m2'].median():,.0f} m² ({df_q['area_km2'].median():.3f} km²)")

# =============================================================================
# RESUMEN DE DATOS PREPARADOS
# =============================================================================

print("\n🎯 RESUMEN DE PREPARACIÓN:")
print(f"   • Muestras por año: {df_m.groupby('ANIO').size().to_dict()}")
print(f"   • Cuadrantes únicos en muestras: {df_m['cod_cuadrante'].nunique()}")
print(f"   • Cuadrantes con 'FUERA': {(df_m['cod_cuadrante'] == 'FUERA').sum():,}")
print(f"   • Cuadrantes en métricas: {len(df_q)}")
print("✅ Datos preparados correctamente")

📂 Cargando datos...
   Leyendo muestras desde: muestras_MANIZALES_2021-2024.csv
   Leyendo métricas desde: manizales_metricas.csv
📊 Muestras cargadas: 26,977 registros
📐 Cuadrantes cargados: 30 cuadrantes
Index(['id', 'fecha_evento', 'id_autor', 'lat', 'lot', 'cod_cuadrante'], dtype='object')
Index(['cod_cuadrante', 'ciudad', 'area_m2', 'perimetro_m', 'centroid_lon', 'centroid_lat', 'compactness_polsby',
       'elongation_ratio', 'holes_count'],
      dtype='object')
📅 Registros válidos en período 2021-2024: 26,977
🗑️ Registros filtrados: 0
📏 Área promedio: 455,453 m² (0.455 km²)
📏 Área mediana: 489,497 m² (0.489 km²)

🎯 RESUMEN DE PREPARACIÓN:
   • Muestras por año: {2021: 8138, 2022: 21, 2023: 8567, 2024: 10251}
   • Cuadrantes únicos en muestras: 31
   • Cuadrantes con 'FUERA': 1,108
   • Cuadrantes en métricas: 30
✅ Datos preparados correctamente


In [7]:
# ================================================================================================
# 3. AGREGACIONES BASE POR CUADRANTE Y AÑO  
# ================================================================================================

print("🔢 Calculando conteos por cuadrante y año...")

# =============================================================================
# CONTEOS POR AÑO USANDO PIVOT TABLE
# =============================================================================

# Crear tabla de conteos: filas=cod_cuadrante, columnas=ANIO, valores=count
conteos_pivot = (df_m.groupby(['cod_cuadrante', 'ANIO'])
                .size()
                .reset_index(name='count')
                .pivot(index='cod_cuadrante', columns='ANIO', values='count')
                .fillna(0)  # Rellenar con 0 donde no hay datos
                .astype(int))

# Renombrar columnas a formato k_YYYY
conteos_pivot.columns = [f'k_{int(year)}' for year in conteos_pivot.columns]

# Asegurar que tenemos todas las columnas de años esperadas
for year in ANIOS:
    col_name = f'k_{year}'
    if col_name not in conteos_pivot.columns:
        conteos_pivot[col_name] = 0

# Reordenar columnas en orden cronológico
year_cols = [f'k_{year}' for year in sorted(ANIOS)]
conteos_pivot = conteos_pivot[year_cols]

print(f"📊 Cuadrantes con datos: {len(conteos_pivot)}")

# =============================================================================
# CÁLCULO DE TOTALES
# =============================================================================

# Calcular k_total = suma de todos los años
conteos_pivot['k_total'] = conteos_pivot[year_cols].sum(axis=1)

# Convertir índice a columna para facilitar merges posteriores
df_conteos = conteos_pivot.reset_index()

print(f"🎯 Total muestras procesadas: {df_conteos['k_total'].sum():,}")
print(f"📈 Distribución por año:")
for year in ANIOS:
    col = f'k_{year}'
    total = df_conteos[col].sum()
    print(f"   • {year}: {total:,} muestras")

# =============================================================================
# VERIFICAR CUADRANTE 'FUERA'
# =============================================================================

fuera_idx = df_conteos['cod_cuadrante'] == 'FUERA'
if fuera_idx.any():
    fuera_total = df_conteos.loc[fuera_idx, 'k_total'].iloc[0]
    print(f"🔍 Registros 'FUERA': {fuera_total:,} ({fuera_total/df_conteos['k_total'].sum()*100:.1f}%)")
else:
    print("ℹ️  No hay registros 'FUERA' (todos tienen cuadrante válido)")

print(f"\n📋 Primeras filas de conteos:")
print(df_conteos.to_string(index=False))
print("✅ Agregaciones base completadas")

🔢 Calculando conteos por cuadrante y año...
📊 Cuadrantes con datos: 31
🎯 Total muestras procesadas: 26,977
📈 Distribución por año:
   • 2021: 8,138 muestras
   • 2022: 21 muestras
   • 2023: 8,567 muestras
   • 2024: 10,251 muestras
🔍 Registros 'FUERA': 1,108 (4.1%)

📋 Primeras filas de conteos:
cod_cuadrante  k_2021  k_2022  k_2023  k_2024  k_total
        FUERA     222      21     395     470     1108
       MZ_001       0       0      62       0       62
       MZ_002       0       0     563       0      563
       MZ_003       1       0     588       0      589
       MZ_004       0       0       0     167      167
       MZ_005     131       0     279      88      498
       MZ_006       0       0       0      39       39
       MZ_007       0       0     226     988     1214
       MZ_008     362       0     199     231      792
       MZ_009       0       0     936    1055     1991
       MZ_010       0       0      58       0       58
       MZ_011       3       0      59      

In [8]:
# ================================================================================================
# 4. DÍAS CON ACTIVIDAD Y ≥2 MUESTRAS POR AÑO
# ================================================================================================

print("📅 Calculando días con actividad...")

# =============================================================================
# PREPARAR DATOS DIARIOS
# =============================================================================

# Crear columna FECHA (solo la parte de fecha, sin hora)
df_m['FECHA'] = df_m['fecha_evento'].dt.date

# Agrupar por cuadrante, año y fecha para contar muestras por día
daily_counts = (df_m.groupby(['cod_cuadrante', 'ANIO', 'FECHA'])
               .size()
               .reset_index(name='muestras_dia'))

print(f"📈 Días únicos con datos: {len(daily_counts):,}")

# =============================================================================
# CALCULAR DÍAS CON ACTIVIDAD (≥1 muestra)
# =============================================================================

# Días con al menos 1 muestra por cuadrante y año
dias_any = (daily_counts.groupby(['cod_cuadrante', 'ANIO'])
           .size()  # Cuenta días únicos
           .reset_index(name='dias_any')
           .pivot(index='cod_cuadrante', columns='ANIO', values='dias_any')
           .fillna(0)
           .astype(int))

# Renombrar columnas
dias_any.columns = [f'dias_any_{int(year)}' for year in dias_any.columns]

# Asegurar todas las columnas de años
for year in ANIOS:
    col_name = f'dias_any_{year}'
    if col_name not in dias_any.columns:
        dias_any[col_name] = 0

# =============================================================================
# CALCULAR DÍAS CON ≥2 MUESTRAS
# =============================================================================

# Filtrar días con 2+ muestras
daily_ge2 = daily_counts[daily_counts['muestras_dia'] >= 2]

# Contar días ≥2 por cuadrante y año
dias_ge2 = (daily_ge2.groupby(['cod_cuadrante', 'ANIO'])
           .size()
           .reset_index(name='dias_ge2')
           .pivot(index='cod_cuadrante', columns='ANIO', values='dias_ge2')
           .fillna(0)
           .astype(int))

# Renombrar columnas
if len(dias_ge2.columns) > 0:
    dias_ge2.columns = [f'dias_ge2_{int(year)}' for year in dias_ge2.columns]

# Asegurar todas las columnas de años
for year in ANIOS:
    col_name = f'dias_ge2_{year}'
    if col_name not in dias_ge2.columns:
        dias_ge2[col_name] = 0

# =============================================================================
# CALCULAR TOTALES 4 AÑOS
# =============================================================================

# Totales de días any
dias_any_cols = [f'dias_any_{year}' for year in ANIOS]
dias_any['dias_any_total'] = dias_any[dias_any_cols].sum(axis=1)

# Totales de días ≥2
dias_ge2_cols = [f'dias_ge2_{year}' for year in ANIOS]  
dias_ge2['dias_ge2_total'] = dias_ge2[dias_ge2_cols].sum(axis=1)

# =============================================================================
# MERGE DE DÍAS CON CONTEOS PARA CALCULAR PROMEDIOS
# =============================================================================

# Preparar dataframes para merge
dias_any_df = dias_any.reset_index()
dias_ge2_df = dias_ge2.reset_index()

# Merge con conteos
df_combined = df_conteos.merge(dias_any_df, on='cod_cuadrante', how='left')
df_combined = df_combined.merge(dias_ge2_df, on='cod_cuadrante', how='left')

# Rellenar NaN con 0 en días
dias_cols = [col for col in df_combined.columns if col.startswith('dias_')]
df_combined[dias_cols] = df_combined[dias_cols].fillna(0).astype(int)

# =============================================================================
# CALCULAR PROMEDIOS DIARIOS
# =============================================================================

# Promedios diarios por año (any activity)
for year in ANIOS:
    k_col = f'k_{year}'
    dias_col = f'dias_any_{year}'
    avg_col = f'avg_dia_any_{year}'
    
    # avg = k / dias, pero solo donde dias > 0
    df_combined[avg_col] = np.where(
        df_combined[dias_col] > 0,
        df_combined[k_col] / df_combined[dias_col],
        np.nan
    )

# Promedios diarios por año (≥2 activity)  
for year in ANIOS:
    k_col = f'k_{year}'
    dias_col = f'dias_ge2_{year}'
    avg_col = f'avg_dia_ge2_{year}'
    
    df_combined[avg_col] = np.where(
        df_combined[dias_col] > 0,
        df_combined[k_col] / df_combined[dias_col],
        np.nan
    )

# Promedios totales
df_combined['avg_dia_any_total'] = np.where(
    df_combined['dias_any_total'] > 0,
    df_combined['k_total'] / df_combined['dias_any_total'],
    np.nan
)

df_combined['avg_dia_ge2_total'] = np.where(
    df_combined['dias_ge2_total'] > 0,
    df_combined['k_total'] / df_combined['dias_ge2_total'],
    np.nan
)

# =============================================================================
# RESUMEN DE DÍAS
# =============================================================================

print("📊 Resumen de días con actividad:")
for year in ANIOS:
    total_dias_any = df_combined[f'dias_any_{year}'].sum()
    total_dias_ge2 = df_combined[f'dias_ge2_{year}'].sum()
    print(f"   • {year}: {total_dias_any:,} días (≥1), {total_dias_ge2:,} días (≥2)")

print(f"\n📈 Promedios diarios globales:")
for year in ANIOS:
    avg_any = df_combined[f'avg_dia_any_{year}'].mean()
    avg_ge2 = df_combined[f'avg_dia_ge2_{year}'].mean()  
    print(f"   • {year}: {avg_any:.2f} muestras/día (≥1), {avg_ge2:.2f} muestras/día (≥2)")

print("✅ Cálculos de días completados")

📅 Calculando días con actividad...
📈 Días únicos con datos: 1,300
📊 Resumen de días con actividad:
   • 2021: 154 días (≥1), 129 días (≥2)
   • 2022: 14 días (≥1), 4 días (≥2)
   • 2023: 620 días (≥1), 454 días (≥2)
   • 2024: 512 días (≥1), 413 días (≥2)

📈 Promedios diarios globales:
   • 2021: 63.70 muestras/día (≥1), 74.04 muestras/día (≥2)
   • 2022: 1.50 muestras/día (≥1), 5.25 muestras/día (≥2)
   • 2023: 17.19 muestras/día (≥1), 21.60 muestras/día (≥2)
   • 2024: 27.37 muestras/día (≥1), 30.37 muestras/día (≥2)
✅ Cálculos de días completados


In [9]:
# ================================================================================================
# 5. MERGE CON ÁREAS DE CUADRANTES
# ================================================================================================

print("📐 Incorporando áreas de cuadrantes...")

# =============================================================================
# PREPARAR TABLA DE ÁREAS
# =============================================================================

# Seleccionar columnas de área y remover duplicados
areas = df_q[['cod_cuadrante', 'area_m2', 'area_km2']].drop_duplicates('cod_cuadrante')

print(f"📏 Cuadrantes con área disponible: {len(areas):,}")
print(f"📏 Área total disponible: {areas['area_km2'].sum():.2f} km²")

# =============================================================================
# MERGE CON DATOS AGREGADOS (LEFT JOIN)
# =============================================================================

# Hacer left join para mantener todos los cuadrantes de los datos
df_final = df_combined.merge(areas, on='cod_cuadrante', how='left')

# Verificar resultado del merge
cuadrantes_sin_area = df_final['area_m2'].isna().sum()
cuadrantes_con_area = df_final['area_m2'].notna().sum()

print(f"🔗 Merge completado:")
print(f"   • Cuadrantes con área: {cuadrantes_con_area:,}")
print(f"   • Cuadrantes sin área: {cuadrantes_sin_area:,}")

# =============================================================================
# VERIFICAR CUADRANTE 'FUERA'
# =============================================================================

fuera_mask = df_final['cod_cuadrante'] == 'FUERA'
if fuera_mask.any():
    fuera_area = df_final.loc[fuera_mask, 'area_m2'].iloc[0]
    if pd.isna(fuera_area):
        print("✅ Cuadrante 'FUERA' correctamente sin área (NaN)")
    else:
        print(f"⚠️  Cuadrante 'FUERA' tiene área: {fuera_area} m²")

# =============================================================================
# ESTADÍSTICAS DE ÁREAS
# =============================================================================

areas_validas = df_final[df_final['area_m2'] > 0]['area_m2']
if len(areas_validas) > 0:
    print(f"\n📊 Estadísticas de áreas (cuadrantes válidos):")
    print(f"   • Mínima: {areas_validas.min():,.0f} m² ({areas_validas.min()/1e6:.4f} km²)")
    print(f"   • Mediana: {areas_validas.median():,.0f} m² ({areas_validas.median()/1e6:.3f} km²)")
    print(f"   • Máxima: {areas_validas.max():,.0f} m² ({areas_validas.max()/1e6:.3f} km²)")
    print(f"   • Promedio: {areas_validas.mean():,.0f} m² ({areas_validas.mean()/1e6:.3f} km²)")

# =============================================================================
# LISTAR CUADRANTES SIN ÁREA (PARA QA)
# =============================================================================

cuadrantes_sin_area_list = df_final[df_final['area_m2'].isna()]['cod_cuadrante'].tolist()
if cuadrantes_sin_area_list:
    print(f"\n⚠️  Cuadrantes sin área ({len(cuadrantes_sin_area_list)}):")
    for i, cod in enumerate(cuadrantes_sin_area_list[:10]):  # Mostrar máximo 10
        muestras = df_final[df_final['cod_cuadrante'] == cod]['k_total'].iloc[0]
        print(f"   • {cod}: {muestras} muestras")
    if len(cuadrantes_sin_area_list) > 10:
        print(f"   • ... y {len(cuadrantes_sin_area_list) - 10} más")

print("✅ Incorporación de áreas completada")

📐 Incorporando áreas de cuadrantes...
📏 Cuadrantes con área disponible: 30
📏 Área total disponible: 13.66 km²
🔗 Merge completado:
   • Cuadrantes con área: 30
   • Cuadrantes sin área: 1
✅ Cuadrante 'FUERA' correctamente sin área (NaN)

📊 Estadísticas de áreas (cuadrantes válidos):
   • Mínima: 14,340 m² (0.0143 km²)
   • Mediana: 489,497 m² (0.489 km²)
   • Máxima: 1,287,667 m² (1.288 km²)
   • Promedio: 455,453 m² (0.455 km²)

⚠️  Cuadrantes sin área (1):
   • FUERA: 1108 muestras
✅ Incorporación de áreas completada


In [11]:
# ================================================================================================
# 6. DENSIDADES NAÏVE (SIN SUAVIZADO)
# ================================================================================================

print("🎯 Calculando densidades naïve...")

# =============================================================================
# CALCULAR DENSIDADES SIMPLES
# =============================================================================

# Crear mask para cuadrantes con área válida
area_valida = (df_final['area_m2'].notna()) & (df_final['area_m2'] > 0)

# Inicializar columnas de densidad con NaN
df_final['density_naive_per_m2'] = np.nan
df_final['density_naive_per_km2'] = np.nan

# Calcular densidades solo para cuadrantes con área válida
df_final.loc[area_valida, 'density_naive_per_m2'] = (
    df_final.loc[area_valida, 'k_total'] / df_final.loc[area_valida, 'area_m2']
)

df_final.loc[area_valida, 'density_naive_per_km2'] = (
    df_final.loc[area_valida, 'k_total'] / df_final.loc[area_valida, 'area_km2']
)

# =============================================================================
# ESTADÍSTICAS DE DENSIDADES NAÏVE
# =============================================================================

densidades_validas = df_final[area_valida]['density_naive_per_km2']
n_cuadrantes_con_densidad = len(densidades_validas)

print(f"📊 Densidades calculadas para {n_cuadrantes_con_densidad:,} cuadrantes")

if n_cuadrantes_con_densidad > 0:
    print(f"\n📈 Estadísticas de densidad naïve (muestras/km²):")
    print(f"   • Mínima: {densidades_validas.min():.4f}")
    print(f"   • Cuartil 1: {densidades_validas.quantile(0.25):.3f}")
    print(f"   • Mediana: {densidades_validas.median():.3f}")
    print(f"   • Cuartil 3: {densidades_validas.quantile(0.75):.3f}")
    print(f"   • Máxima: {densidades_validas.max():.2f}")
    print(f"   • Promedio: {densidades_validas.mean():.3f}")
    print(f"   • Desv. Estándar: {densidades_validas.std():.3f}")
    
    # Cuadrantes con alta densidad
    alta_densidad = densidades_validas > densidades_validas.quantile(0.95)
    n_alta_densidad = alta_densidad.sum()
    print(f"   • Cuadrantes > P95: {n_alta_densidad}")

# =============================================================================
# IDENTIFICAR CUADRANTES CON DENSIDAD EXTREMA
# =============================================================================

if n_cuadrantes_con_densidad > 0:
    # Top 5 densidades más altas
    top_densidades = df_final[area_valida].nlargest(5, 'density_naive_per_km2')
    
    print(f"\n🔝 Top 5 cuadrantes con mayor densidad:")
    for idx, row in top_densidades.iterrows():
        print(f"   • {row['cod_cuadrante']}: {row['density_naive_per_km2']:.2f} muestras/km² "
              f"({row['k_total']} muestras, {row['area_km2']:.4f} km²)")

# =============================================================================
# CUADRANTES SIN MUESTRAS (DENSIDAD = 0)
# =============================================================================

sin_muestras = df_final[(area_valida) & (df_final['k_total'] == 0)]
n_sin_muestras = len(sin_muestras)

print(f"\n🔍 Cuadrantes sin muestras: {n_sin_muestras:,}")

if n_sin_muestras > 0 and n_sin_muestras <= 20:
    print(f"   Listado completo:")
    for idx, row in sin_muestras.iterrows():
        print(f"   • {row['cod_cuadrante']}: 0 muestras ({row['area_km2']:.4f} km²)")
elif n_sin_muestras > 20:
    print(f"   (Lista muy larga, mostrando solo algunos ejemplos)")
    for idx, row in sin_muestras.head(10).iterrows():
        print(f"   • {row['cod_cuadrante']}: 0 muestras ({row['area_km2']:.4f} km²)")
    print(f"   ... y {n_sin_muestras - 10} más")

# =============================================================================
# VALIDACIÓN DE DENSIDADES
# =============================================================================

# Verificar que no hay densidades negativas
densidades_negativas = (df_final['density_naive_per_km2'] < 0).sum()
if densidades_negativas > 0:
    print(f"⚠️  ADVERTENCIA: {densidades_negativas} densidades negativas detectadas")

# Verificar consistencia entre densidades por m² y km²
if n_cuadrantes_con_densidad > 0:
    ratio_check = df_final[area_valida]['density_naive_per_km2'] / (df_final[area_valida]['density_naive_per_m2'] * 1e6)
    ratio_ok = np.allclose(ratio_check, 1.0, rtol=1e-6)
    if ratio_ok:
        print("✅ Consistencia entre densidades por m² y km² verificada")
    else:
        print("⚠️  ADVERTENCIA: Inconsistencia en conversión m²/km²")

print("✅ Cálculo de densidades naïve completado")

🎯 Calculando densidades naïve...
📊 Densidades calculadas para 30 cuadrantes

📈 Estadísticas de densidad naïve (muestras/km²):
   • Mínima: 64.6833
   • Cuartil 1: 637.160
   • Mediana: 1343.869
   • Cuartil 3: 3400.431
   • Máxima: 4957.96
   • Promedio: 1951.233
   • Desv. Estándar: 1609.788
   • Cuadrantes > P95: 2

🔝 Top 5 cuadrantes con mayor densidad:
   • MZ_012: 4957.96 muestras/km² (3181 muestras, 0.6416 km²)
   • MZ_030: 4463.00 muestras/km² (64 muestras, 0.0143 km²)
   • MZ_028: 4280.63 muestras/km² (2131 muestras, 0.4978 km²)
   • MZ_018: 4189.65 muestras/km² (1737 muestras, 0.4146 km²)
   • MZ_008: 3875.13 muestras/km² (792 muestras, 0.2044 km²)

🔍 Cuadrantes sin muestras: 0
✅ Consistencia entre densidades por m² y km² verificada
✅ Cálculo de densidades naïve completado


In [14]:
# ================================================================================================
# 7. FUNCIONES IDBA (EMPIRICAL BAYES POISSON-GAMMA)
# ================================================================================================

print("🧠 Definiendo funciones IDBA con metodología Poisson-Gamma...")

def estimate_gamma_parameters(counts, areas, method='mom'):
    """
    Estima parámetros α y β de la distribución Gamma prior usando Empirical Bayes.
    
    Metodología:
    - Asume que λᵢ ~ Gamma(α, β) donde λᵢ es la tasa real del cuadrante i
    - Observamos kᵢ ~ Poisson(λᵢ × Aᵢ) donde Aᵢ es el área del cuadrante
    - La distribución marginal es k ~ Negative Binomial
    
    Parameters:
    -----------
    counts : array-like
        Conteos observados (k_total)
    areas : array-like  
        Áreas correspondientes (area_km2)
    method : str
        Método de estimación: 'mom' (method of moments) o 'mle' (maximum likelihood)
        
    Returns:
    --------
    dict : {'alpha': float, 'beta': float, 'converged': bool, 'iterations': int}
    """
    
    # Convertir a numpy arrays y filtrar valores válidos
    k = np.array(counts, dtype=float)
    A = np.array(areas, dtype=float)
    
    # Filtrar NaN, negativos y ceros en áreas
    valid_mask = (~np.isnan(k)) & (~np.isnan(A)) & (A > 0) & (k >= 0)
    k_valid = k[valid_mask]
    A_valid = A[valid_mask]
    
    n = len(k_valid)
    if n < 2:
        return {'alpha': 1.0, 'beta': 1.0, 'converged': False, 'iterations': 0, 'error': 'Datos insuficientes'}
    
    # Calcular tasas observadas (MLE individual)
    rates_observed = k_valid / A_valid
    
    if method == 'mom':
        # Method of Moments (más robusto)
        mean_rate = np.mean(rates_observed)
        var_rate = np.var(rates_observed, ddof=1)
        
        if var_rate <= 0 or mean_rate <= 0:
            return {'alpha': 1.0, 'beta': 1.0, 'converged': False, 'iterations': 0, 'error': 'Varianza o media no válida'}
        
        # Estimadores MoM para Gamma
        alpha_hat = (mean_rate ** 2) / var_rate
        beta_hat = mean_rate / var_rate
        
        return {
            'alpha': max(0.1, alpha_hat),  # Evitar valores muy pequeños
            'beta': max(0.1, beta_hat),
            'converged': True,
            'iterations': 1,
            'method': 'Method of Moments'
        }
    
    elif method == 'mle':
        # Maximum Likelihood Estimation (más preciso pero puede no converger)
        from scipy.optimize import minimize_scalar
        from scipy.special import digamma
        
        def neg_log_likelihood(alpha):
            # Para α dado, β = α / mean_rate
            mean_rate = np.mean(rates_observed)
            beta = alpha / mean_rate if mean_rate > 0 else 1.0
            
            if alpha <= 0 or beta <= 0:
                return 1e10
            
            # Log-likelihood de Negative Binomial marginal
            # k_i ~ NB(α, β/(β + A_i))
            ll = 0
            for i in range(n):
                p_i = beta / (beta + A_valid[i])
                if p_i <= 0 or p_i >= 1:
                    return 1e10
                
                # Log P(K = k_i)
                from scipy.special import gammaln
                ll += (gammaln(k_valid[i] + alpha) - gammaln(alpha) - gammaln(k_valid[i] + 1)
                       + alpha * np.log(p_i) + k_valid[i] * np.log(1 - p_i))
            
            return -ll
        
        try:
            # Buscar α óptimo
            result = minimize_scalar(neg_log_likelihood, bounds=(0.1, 100), method='bounded')
            alpha_mle = result.x
            mean_rate = np.mean(rates_observed)
            beta_mle = alpha_mle / mean_rate if mean_rate > 0 else 1.0
            
            return {
                'alpha': alpha_mle,
                'beta': beta_mle,
                'converged': result.success,
                'iterations': result.nfev,
                'method': 'Maximum Likelihood'
            }
        except Exception as e:
            # Fallback a Method of Moments
            print(f"⚠️  MLE falló, usando MoM como respaldo: {str(e)}")
            return estimate_gamma_parameters(counts, areas, method='mom')
    
    else:
        raise ValueError("Method debe ser 'mom' o 'mle'")


def compute_idba_estimates(counts, areas, alpha, beta):
    """
    Calcula estimadores IDBA (Empirical Bayes) usando prior Gamma(α, β).
    
    Metodología:
    - Prior: λᵢ ~ Gamma(α, β)
    - Likelihood: kᵢ|λᵢ ~ Poisson(λᵢ × Aᵢ)
    - Posterior: λᵢ|kᵢ ~ Gamma(α + kᵢ, β + Aᵢ)
    - Estimador EB: E[λᵢ|kᵢ] = (α + kᵢ)/(β + Aᵢ)
    
    Parameters:
    -----------
    counts : array-like
        Conteos observados
    areas : array-like
        Áreas correspondientes
    alpha, beta : float
        Parámetros del prior Gamma
        
    Returns:
    --------
    dict con estimaciones IDBA
    """
    
    k = np.array(counts, dtype=float)
    A = np.array(areas, dtype=float)
    
    # Filtrar valores válidos
    valid_mask = (~np.isnan(k)) & (~np.isnan(A)) & (A > 0) & (k >= 0)
    
    # Inicializar con NaN
    lambda_eb = np.full_like(k, np.nan)
    density_eb = np.full_like(k, np.nan)
    shrinkage_factor = np.full_like(k, np.nan)
    
    # Calcular solo para valores válidos
    k_valid = k[valid_mask]
    A_valid = A[valid_mask]
    
    if len(k_valid) > 0:
        # Estimador Empirical Bayes (media posterior)
        lambda_eb_valid = (alpha + k_valid) / (beta + A_valid)
        
        # Densidad EB (es la misma que λ porque densidad = λ)
        density_eb_valid = lambda_eb_valid
        
        # Factor de shrinkage: qué tanto se contrae hacia el prior
        # shrinkage = β/(β + Aᵢ), valores altos indican más shrinkage
        shrinkage_valid = beta / (beta + A_valid)
        
        # Asignar valores calculados
        lambda_eb[valid_mask] = lambda_eb_valid
        density_eb[valid_mask] = density_eb_valid
        shrinkage_factor[valid_mask] = shrinkage_valid
    
    return {
        'lambda_eb': lambda_eb,
        'density_eb_per_km2': density_eb,
        'shrinkage_factor': shrinkage_factor,
        'n_valid': len(k_valid),
        'alpha_used': alpha,
        'beta_used': beta
    }


def calculate_credible_intervals(counts, areas, alpha, beta, confidence=0.95):
    """
    Calcula intervalos de credibilidad para las tasas λᵢ usando la posterior Gamma.
    
    Parameters:
    -----------
    counts, areas : array-like
        Datos observados
    alpha, beta : float
        Parámetros prior
    confidence : float
        Nivel de confianza (default 0.95 para 95%)
        
    Returns:
    --------
    dict con límites inferior y superior
    """
    from scipy.stats import gamma
    
    k = np.array(counts, dtype=float)
    A = np.array(areas, dtype=float)
    
    valid_mask = (~np.isnan(k)) & (~np.isnan(A)) & (A > 0) & (k >= 0)
    
    lower = np.full_like(k, np.nan)
    upper = np.full_like(k, np.nan)
    
    k_valid = k[valid_mask]
    A_valid = A[valid_mask]
    
    if len(k_valid) > 0:
        # Parámetros posteriores
        alpha_post = alpha + k_valid
        beta_post = beta + A_valid
        
        # Cuantiles de la distribución posterior
        alpha_level = (1 - confidence) / 2
        lower_valid = gamma.ppf(alpha_level, alpha_post, scale=1/beta_post)
        upper_valid = gamma.ppf(1 - alpha_level, alpha_post, scale=1/beta_post)
        
        lower[valid_mask] = lower_valid
        upper[valid_mask] = upper_valid
    
    return {
        'lower': lower,
        'upper': upper,
        'confidence': confidence
    }

print("✅ Funciones IDBA definidas correctamente")

🧠 Definiendo funciones IDBA con metodología Poisson-Gamma...
✅ Funciones IDBA definidas correctamente


In [15]:
# ================================================================================================
# DIAGNÓSTICO DEL ERROR EN IDBA
# ================================================================================================

print("🔍 Diagnosticando el error de longitud de arrays...")

# Verificar las longitudes de los componentes
print(f"📊 Longitud de df_final: {len(df_final)}")
print(f"📊 Longitud de df_final['k_total']: {len(df_final['k_total'])}")
print(f"📊 Longitud de df_final['area_km2']: {len(df_final['area_km2'])}")

# Verificar máscara de valores válidos para IDBA
idba_mask = (df_final['area_km2'].notna()) & (df_final['area_km2'] > 0) & (df_final['k_total'] >= 0)
print(f"📊 Cuadrantes elegibles para IDBA: {idba_mask.sum()}")
print(f"📊 Cuadrantes NO elegibles: {(~idba_mask).sum()}")

# Verificar si hay algún problema en los datos
print(f"\n🔍 Detalle de valores:")
print(f"   • k_total con NaN: {df_final['k_total'].isna().sum()}")  
print(f"   • area_km2 con NaN: {df_final['area_km2'].isna().sum()}")
print(f"   • area_km2 <= 0: {(df_final['area_km2'] <= 0).sum()}")
print(f"   • k_total < 0: {(df_final['k_total'] < 0).sum()}")

# Mostrar algunos ejemplos de registros problemáticos
print(f"\n📋 Registros con área NaN:")
area_nan = df_final[df_final['area_km2'].isna()]
if len(area_nan) > 0:
    print(area_nan[['cod_cuadrante', 'k_total', 'area_km2']].head())

print(f"\n📋 Registros con área <= 0:")
area_zero = df_final[df_final['area_km2'] <= 0]
if len(area_zero) > 0:
    print(area_zero[['cod_cuadrante', 'k_total', 'area_km2']].head())

🔍 Diagnosticando el error de longitud de arrays...
📊 Longitud de df_final: 31
📊 Longitud de df_final['k_total']: 31
📊 Longitud de df_final['area_km2']: 31
📊 Cuadrantes elegibles para IDBA: 30
📊 Cuadrantes NO elegibles: 1

🔍 Detalle de valores:
   • k_total con NaN: 0
   • area_km2 con NaN: 1
   • area_km2 <= 0: 0
   • k_total < 0: 0

📋 Registros con área NaN:
  cod_cuadrante  k_total  area_km2
0         FUERA     1108       NaN

📋 Registros con área <= 0:


In [16]:
# ================================================================================================
# 8. APLICAR IDBA Y ENSAMBLAR TABLA FINAL
# ================================================================================================

print("🔥 Aplicando metodología IDBA Poisson-Gamma...")

# =============================================================================
# PREPARAR DATOS PARA IDBA
# =============================================================================

# Filtrar cuadrantes con datos válidos para IDBA
idba_mask = (df_final['area_km2'].notna()) & (df_final['area_km2'] > 0) & (df_final['k_total'] >= 0)
idba_data = df_final[idba_mask].copy()

print(f"📊 Cuadrantes elegibles para IDBA: {len(idba_data):,}")
print(f"📊 Cuadrantes excluidos (sin área): {(~idba_mask).sum():,}")

if len(idba_data) == 0:
    print("⚠️  ADVERTENCIA: No hay datos válidos para aplicar IDBA")
else:
    # =============================================================================
    # ESTIMAR PARÁMETROS DEL PRIOR GAMMA
    # =============================================================================
    
    print("\n🎯 Estimando parámetros del prior Gamma...")
    
    # Probar ambos métodos
    params_mom = estimate_gamma_parameters(
        idba_data['k_total'], 
        idba_data['area_km2'], 
        method='mom'
    )
    
    params_mle = estimate_gamma_parameters(
        idba_data['k_total'], 
        idba_data['area_km2'], 
        method='mle'
    )
    
    print(f"📈 Method of Moments:")
    print(f"   • α = {params_mom['alpha']:.4f}")
    print(f"   • β = {params_mom['beta']:.4f}")
    print(f"   • Convergió: {params_mom['converged']}")
    
    print(f"📈 Maximum Likelihood:")
    print(f"   • α = {params_mle['alpha']:.4f}")
    print(f"   • β = {params_mle['beta']:.4f}")
    print(f"   • Convergió: {params_mle['converged']}")
    
    # Seleccionar parámetros (preferir MLE si convergió, sino MoM)
    if params_mle['converged']:
        alpha_final = params_mle['alpha']
        beta_final = params_mle['beta']
        metodo_usado = "MLE"
    else:
        alpha_final = params_mom['alpha']
        beta_final = params_mom['beta']
        metodo_usado = "MoM"
    
    print(f"\n🎯 Parámetros finales (método {metodo_usado}):")
    print(f"   • α = {alpha_final:.4f}")
    print(f"   • β = {beta_final:.4f}")
    
    # Interpretación de los parámetros
    media_prior = alpha_final / beta_final
    var_prior = alpha_final / (beta_final ** 2)
    cv_prior = np.sqrt(var_prior) / media_prior
    
    print(f"   • Media prior: {media_prior:.4f} muestras/km²")
    print(f"   • Desv. Estándar prior: {np.sqrt(var_prior):.4f}")
    print(f"   • Coef. Variación prior: {cv_prior:.3f}")

# =============================================================================
# APLICAR IDBA A TODOS LOS DATOS
# =============================================================================

print(f"\n🧠 Aplicando Empirical Bayes...")

# Calcular estimaciones IDBA
idba_results = compute_idba_estimates(
    df_final['k_total'], 
    df_final['area_km2'], 
    alpha_final, 
    beta_final
)

# Agregar columnas IDBA al dataframe
df_final['density_idba_per_km2'] = idba_results['density_eb_per_km2']
df_final['shrinkage_factor'] = idba_results['shrinkage_factor']

print(f"✅ IDBA aplicado a {idba_results['n_valid']:,} cuadrantes")

# =============================================================================
# CALCULAR INTERVALOS DE CREDIBILIDAD
# =============================================================================

print("📊 Calculando intervalos de credibilidad...")

credible_95 = calculate_credible_intervals(
    df_final['k_total'], 
    df_final['area_km2'], 
    alpha_final, 
    beta_final,
    confidence=0.95
)

df_final['density_idba_lower_95'] = credible_95['lower']
df_final['density_idba_upper_95'] = credible_95['upper']

# Calcular ancho del intervalo
df_final['credible_interval_width'] = (
    df_final['density_idba_upper_95'] - df_final['density_idba_lower_95']
)

print("✅ Intervalos de credibilidad calculados")

# =============================================================================
# ESTADÍSTICAS DE RESULTADOS IDBA
# =============================================================================

valid_idba = df_final['density_idba_per_km2'].notna()
densidades_idba = df_final[valid_idba]['density_idba_per_km2']

if len(densidades_idba) > 0:
    print(f"\n📈 Estadísticas densidades IDBA:")
    print(f"   • N válidas: {len(densidades_idba):,}")
    print(f"   • Mínima: {densidades_idba.min():.4f}")
    print(f"   • Mediana: {densidades_idba.median():.3f}")
    print(f"   • Máxima: {densidades_idba.max():.3f}")
    print(f"   • Promedio: {densidades_idba.mean():.3f}")
    print(f"   • Desv. Estándar: {densidades_idba.std():.3f}")

# =============================================================================
# ANÁLISIS DE SHRINKAGE
# =============================================================================

valid_shrinkage = df_final['shrinkage_factor'].notna()
shrinkage_vals = df_final[valid_shrinkage]['shrinkage_factor']

if len(shrinkage_vals) > 0:
    print(f"\n🔄 Análisis de shrinkage:")
    print(f"   • Shrinkage mínimo: {shrinkage_vals.min():.3f}")
    print(f"   • Shrinkage mediano: {shrinkage_vals.median():.3f}")
    print(f"   • Shrinkage máximo: {shrinkage_vals.max():.3f}")
    print(f"   • Shrinkage promedio: {shrinkage_vals.mean():.3f}")
    
    # Cuadrantes con mucho shrinkage (área pequeña)
    alto_shrinkage = shrinkage_vals > 0.8
    print(f"   • Cuadrantes con shrinkage > 80%: {alto_shrinkage.sum():,}")

# =============================================================================
# COMPARACIÓN NAÏVE vs IDBA
# =============================================================================

both_valid = valid_idba & df_final['density_naive_per_km2'].notna()
if both_valid.sum() > 0:
    naive_vs_idba = df_final[both_valid][['density_naive_per_km2', 'density_idba_per_km2']]
    correlacion = naive_vs_idba['density_naive_per_km2'].corr(naive_vs_idba['density_idba_per_km2'])
    
    print(f"\n🔗 Comparación Naïve vs IDBA:")
    print(f"   • Correlación: {correlacion:.4f}")
    
    # Diferencia relativa promedio
    diff_rel = abs(naive_vs_idba['density_naive_per_km2'] - naive_vs_idba['density_idba_per_km2']) / (naive_vs_idba['density_naive_per_km2'] + 1e-10)
    print(f"   • Diferencia relativa promedio: {diff_rel.mean():.3f}")

print("\n✅ Análisis IDBA completado")

🔥 Aplicando metodología IDBA Poisson-Gamma...
📊 Cuadrantes elegibles para IDBA: 30
📊 Cuadrantes excluidos (sin área): 1

🎯 Estimando parámetros del prior Gamma...
📈 Method of Moments:
   • α = 1.4692
   • β = 0.1000
   • Convergió: True
📈 Maximum Likelihood:
   • α = 0.9574
   • β = 0.0005
   • Convergió: True

🎯 Parámetros finales (método MLE):
   • α = 0.9574
   • β = 0.0005
   • Media prior: 1951.2326 muestras/km²
   • Desv. Estándar prior: 1994.1711
   • Coef. Variación prior: 1.022

🧠 Aplicando Empirical Bayes...
✅ IDBA aplicado a 30 cuadrantes
📊 Calculando intervalos de credibilidad...
✅ Intervalos de credibilidad calculados

📈 Estadísticas densidades IDBA:
   • N válidas: 30
   • Mínima: 66.1790
   • Mediana: 1345.361
   • Máxima: 4955.658
   • Promedio: 1949.037
   • Desv. Estándar: 1602.881

🔄 Análisis de shrinkage:
   • Shrinkage mínimo: 0.000
   • Shrinkage mediano: 0.001
   • Shrinkage máximo: 0.033
   • Shrinkage promedio: 0.003
   • Cuadrantes con shrinkage > 80%: 0

🔗 Co

In [17]:
# ================================================================================================
# 9. CONTROL DE CALIDAD Y VALIDACIONES
# ================================================================================================

print("🔍 Ejecutando controles de calidad...")

# =============================================================================
# VALIDACIONES BÁSICAS DE DATOS
# =============================================================================

print("1️⃣ Validaciones básicas...")

# Verificar integridad de datos
n_total = len(df_final)
n_con_codigo = df_final['cod_cuadrante'].notna().sum()
n_con_muestras = df_final['k_total'].notna().sum()
n_con_area = df_final['area_km2'].notna().sum()

print(f"   • Total de registros: {n_total:,}")
print(f"   • Con código de cuadrante: {n_con_codigo:,} ({n_con_codigo/n_total*100:.1f}%)")
print(f"   • Con conteo de muestras: {n_con_muestras:,} ({n_con_muestras/n_total*100:.1f}%)")
print(f"   • Con área válida: {n_con_area:,} ({n_con_area/n_total*100:.1f}%)")

# Verificar códigos de cuadrante únicos
codigos_unicos = df_final['cod_cuadrante'].nunique()
if codigos_unicos == n_total:
    print("   ✅ Todos los códigos de cuadrante son únicos")
else:
    duplicados = n_total - codigos_unicos
    print(f"   ⚠️  ADVERTENCIA: {duplicados} códigos de cuadrante duplicados")

# =============================================================================
# VALIDACIONES DE CONTEOS Y ÁREAS
# =============================================================================

print("\n2️⃣ Validaciones de conteos y áreas...")

# Verificar conteos no negativos
conteos_negativos = (df_final['k_total'] < 0).sum()
if conteos_negativos == 0:
    print("   ✅ Todos los conteos son no-negativos")
else:
    print(f"   ⚠️  ADVERTENCIA: {conteos_negativos} conteos negativos")

# Verificar áreas positivas (donde no son NaN)
areas_no_na = df_final['area_km2'].notna()
areas_no_positivas = ((df_final['area_km2'] <= 0) & areas_no_na).sum()
if areas_no_positivas == 0:
    print("   ✅ Todas las áreas no-NaN son positivas")
else:
    print(f"   ⚠️  ADVERTENCIA: {areas_no_positivas} áreas no positivas")

# Estadísticas de años de actividad
if 'dias_activos' in df_final.columns:
    dias_stats = df_final['dias_activos'].describe()
    print(f"\n   📊 Días de actividad:")
    print(f"      • Mín: {dias_stats['min']:.0f} días")
    print(f"      • Mediana: {dias_stats['50%']:.0f} días")  
    print(f"      • Máx: {dias_stats['max']:.0f} días")
    
    # Verificar rangos razonables (2021-2024 = max ~1460 días)
    dias_invalidos = (df_final['dias_activos'] > 1500) | (df_final['dias_activos'] < 0)
    if dias_invalidos.sum() > 0:
        print(f"   ⚠️  ADVERTENCIA: {dias_invalidos.sum()} valores de días fuera de rango")

# =============================================================================
# VALIDACIONES DE DENSIDADES
# =============================================================================

print("\n3️⃣ Validaciones de densidades...")

# Verificar densidades naïve no negativas
if 'density_naive_per_km2' in df_final.columns:
    naive_validas = df_final['density_naive_per_km2'].notna()
    naive_negativas = ((df_final['density_naive_per_km2'] < 0) & naive_validas).sum()
    if naive_negativas == 0:
        print("   ✅ Todas las densidades naïve son no-negativas")
    else:
        print(f"   ⚠️  ADVERTENCIA: {naive_negativas} densidades naïve negativas")

# Verificar densidades IDBA no negativas  
if 'density_idba_per_km2' in df_final.columns:
    idba_validas = df_final['density_idba_per_km2'].notna()
    idba_negativas = ((df_final['density_idba_per_km2'] < 0) & idba_validas).sum()
    if idba_negativas == 0:
        print("   ✅ Todas las densidades IDBA son no-negativas")
    else:
        print(f"   ⚠️  ADVERTENCIA: {idba_negativas} densidades IDBA negativas")

# Verificar intervalos de credibilidad válidos
if 'density_idba_lower_95' in df_final.columns and 'density_idba_upper_95' in df_final.columns:
    intervalos_validos = (
        df_final['density_idba_lower_95'].notna() & 
        df_final['density_idba_upper_95'].notna()
    )
    intervalos_coherentes = (
        df_final['density_idba_lower_95'] <= df_final['density_idba_upper_95']
    )[intervalos_validos]
    
    if intervalos_coherentes.all():
        print("   ✅ Todos los intervalos de credibilidad son coherentes")
    else:
        incoherentes = (~intervalos_coherentes).sum()
        print(f"   ⚠️  ADVERTENCIA: {incoherentes} intervalos incoherentes")

# =============================================================================
# DETECCIÓN DE OUTLIERS
# =============================================================================

print("\n4️⃣ Detección de outliers...")

if 'density_naive_per_km2' in df_final.columns:
    densidades_validas = df_final[df_final['density_naive_per_km2'].notna()]['density_naive_per_km2']
    
    if len(densidades_validas) > 10:
        Q1 = densidades_validas.quantile(0.25)
        Q3 = densidades_validas.quantile(0.75)
        IQR = Q3 - Q1
        outlier_threshold = Q3 + 1.5 * IQR
        
        outliers_leves = (densidades_validas > outlier_threshold).sum()
        outliers_extremos = (densidades_validas > Q3 + 3 * IQR).sum()
        
        print(f"   📊 Outliers en densidad naïve:")
        print(f"      • Leves (> Q3 + 1.5×IQR): {outliers_leves}")
        print(f"      • Extremos (> Q3 + 3×IQR): {outliers_extremos}")
        
        if outliers_extremos > 0:
            top_outliers = densidades_validas.nlargest(min(5, outliers_extremos))
            print(f"      • Top outliers: {list(top_outliers.round(2))}")

# =============================================================================
# RESUMEN DE COMPLETITUD
# =============================================================================

print("\n5️⃣ Resumen de completitud de datos...")

columnas_clave = [
    'cod_cuadrante', 'k_total', 'area_km2', 
    'density_naive_per_km2', 'density_idba_per_km2'
]

for col in columnas_clave:
    if col in df_final.columns:
        completitud = df_final[col].notna().sum()
        porcentaje = completitud / len(df_final) * 100
        print(f"   • {col}: {completitud:,}/{n_total:,} ({porcentaje:.1f}%)")

# =============================================================================
# VALIDACIÓN CUADRANTE 'FUERA'
# =============================================================================

print("\n6️⃣ Validación especial cuadrante 'FUERA'...")

fuera_rows = df_final[df_final['cod_cuadrante'] == 'FUERA']
if len(fuera_rows) > 0:
    fuera_info = fuera_rows.iloc[0]
    print(f"   • Muestras en 'FUERA': {fuera_info['k_total']}")
    print(f"   • Área de 'FUERA': {fuera_info['area_km2']}")
    print(f"   • Densidad naïve 'FUERA': {fuera_info.get('density_naive_per_km2', 'N/A')}")
    print(f"   • Densidad IDBA 'FUERA': {fuera_info.get('density_idba_per_km2', 'N/A')}")
    
    if pd.isna(fuera_info['area_km2']):
        print("   ✅ Cuadrante 'FUERA' correctamente sin área")
    else:
        print("   ⚠️  Cuadrante 'FUERA' tiene área asignada (revisar)")
else:
    print("   ℹ️  No se encontró cuadrante 'FUERA'")

# =============================================================================
# VERIFICACIONES ESTADÍSTICAS FINALES
# =============================================================================

print("\n7️⃣ Verificaciones estadísticas finales...")

# Verificar conservación de masa (suma total de muestras)
suma_muestras_original = df_final['k_total'].sum()
print(f"   • Total de muestras procesadas: {suma_muestras_original:,}")

# Verificar área total
area_total = df_final[df_final['area_km2'].notna()]['area_km2'].sum()
print(f"   • Área total con datos: {area_total:.2f} km²")

# Densidad promedio ponderada por área
if 'density_idba_per_km2' in df_final.columns:
    mask_completo = (df_final['density_idba_per_km2'].notna() & 
                    df_final['area_km2'].notna() & 
                    (df_final['area_km2'] > 0))
    
    if mask_completo.sum() > 0:
        datos_completos = df_final[mask_completo]
        densidad_ponderada = np.average(
            datos_completos['density_idba_per_km2'], 
            weights=datos_completos['area_km2']
        )
        print(f"   • Densidad IDBA ponderada por área: {densidad_ponderada:.4f} muestras/km²")

print("\n✅ Control de calidad completado")

🔍 Ejecutando controles de calidad...
1️⃣ Validaciones básicas...
   • Total de registros: 31
   • Con código de cuadrante: 31 (100.0%)
   • Con conteo de muestras: 31 (100.0%)
   • Con área válida: 30 (96.8%)
   ✅ Todos los códigos de cuadrante son únicos

2️⃣ Validaciones de conteos y áreas...
   ✅ Todos los conteos son no-negativos
   ✅ Todas las áreas no-NaN son positivas

3️⃣ Validaciones de densidades...
   ✅ Todas las densidades naïve son no-negativas
   ✅ Todas las densidades IDBA son no-negativas
   ✅ Todos los intervalos de credibilidad son coherentes

4️⃣ Detección de outliers...
   📊 Outliers en densidad naïve:
      • Leves (> Q3 + 1.5×IQR): 0
      • Extremos (> Q3 + 3×IQR): 0

5️⃣ Resumen de completitud de datos...
   • cod_cuadrante: 31/31 (100.0%)
   • k_total: 31/31 (100.0%)
   • area_km2: 30/31 (96.8%)
   • density_naive_per_km2: 30/31 (96.8%)
   • density_idba_per_km2: 30/31 (96.8%)

6️⃣ Validación especial cuadrante 'FUERA'...
   • Muestras en 'FUERA': 1108
   • Áre

In [ ]:
# ================================================================================================
# 10. PREPARAR Y EXPORTAR RESULTADOS FINALES
# ================================================================================================

print("💾 Preparando exportación de resultados...")

# =============================================================================
# SELECCIONAR Y ORDENAR COLUMNAS PARA EXPORTACIÓN
# =============================================================================

# Definir columnas para la tabla final
columnas_exportacion = [
    # Identificación
    'cod_cuadrante',
    
    # Datos básicos
    'k_total',
    'dias_activos', 
    'area_m2',
    'area_km2',
    
    # Densidades naïve
    'density_naive_per_m2',
    'density_naive_per_km2', 
    
    # Resultados IDBA
    'density_idba_per_km2',
    'density_idba_lower_95',
    'density_idba_upper_95',
    'credible_interval_width',
    'shrinkage_factor'
]

# Filtrar solo las columnas que existen
columnas_disponibles = [col for col in columnas_exportacion if col in df_final.columns]
print(f"📋 Columnas para exportar: {len(columnas_disponibles)}/{len(columnas_exportacion)}")

# Crear dataframe para exportación
df_export = df_final[columnas_disponibles].copy()

# =============================================================================
# ORDENAR RESULTADOS
# =============================================================================

# Ordenar por densidad IDBA descendente, luego por código de cuadrante
if 'density_idba_per_km2' in df_export.columns:
    df_export = df_export.sort_values(
        ['density_idba_per_km2', 'cod_cuadrante'], 
        ascending=[False, True],
        na_position='last'
    )
else:
    df_export = df_export.sort_values('cod_cuadrante')

print(f"📊 Registros ordenados: {len(df_export):,}")

# =============================================================================
# AGREGAR METADATOS Y RANKINGS
# =============================================================================

# Agregar ranking por densidad IDBA
if 'density_idba_per_km2' in df_export.columns:
    mask_valido = df_export['density_idba_per_km2'].notna()
    df_export.loc[mask_valido, 'rank_density_idba'] = (
        df_export[mask_valido]['density_idba_per_km2'].rank(method='dense', ascending=False)
    )

# Agregar percentiles
if 'density_idba_per_km2' in df_export.columns:
    densidades = df_export['density_idba_per_km2'].dropna()
    if len(densidades) > 0:
        df_export.loc[mask_valido, 'percentile_idba'] = (
            df_export[mask_valido]['density_idba_per_km2'].rank(pct=True) * 100
        )

# =============================================================================
# ESTADÍSTICAS FINALES PARA METADATOS
# =============================================================================

metadata = {
    'fecha_procesamiento': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_cuadrantes': len(df_export),
    'cuadrantes_con_muestras': (df_export['k_total'] > 0).sum() if 'k_total' in df_export.columns else 'N/A',
    'cuadrantes_con_area': df_export['area_km2'].notna().sum() if 'area_km2' in df_export.columns else 'N/A',
    'total_muestras': df_export['k_total'].sum() if 'k_total' in df_export.columns else 'N/A',
    'area_total_km2': df_export['area_km2'].sum() if 'area_km2' in df_export.columns else 'N/A',
    'periodo_datos': '2021-2024',
    'ciudad': 'MANIZALES',
    'metodologia': 'IDBA Poisson-Gamma',
}

if 'density_idba_per_km2' in df_export.columns:
    densidades_validas = df_export['density_idba_per_km2'].dropna()
    if len(densidades_validas) > 0:
        metadata.update({
            'densidad_idba_min': float(densidades_validas.min()),
            'densidad_idba_max': float(densidades_validas.max()),
            'densidad_idba_media': float(densidades_validas.mean()),
            'densidad_idba_mediana': float(densidades_validas.median()),
            'parametro_alpha': float(alpha_final),
            'parametro_beta': float(beta_final),
            'metodo_parametros': metodo_usado
        })

print("\n📈 Estadísticas finales:")
for key, value in metadata.items():
    if isinstance(value, float):
        print(f"   • {key}: {value:.4f}")
    else:
        print(f"   • {key}: {value}")

# =============================================================================
# EXPORTAR ARCHIVOS
# =============================================================================

print(f"\n💾 Exportando archivos...")

# Definir rutas de salida
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
archivo_principal = f"indicadores_muestras_manizales_{timestamp}.csv"
archivo_metadatos = f"tools/metadatos_muestras_manizales_{timestamp}.json"

try:
    # Exportar tabla principal
    df_export.to_csv(archivo_principal, index=False, encoding='utf-8-sig', sep=';')
    print(f"   ✅ Tabla principal: {archivo_principal}")
    
    # Exportar metadatos
    import json
    with open(archivo_metadatos, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
    print(f"   ✅ Metadatos: {archivo_metadatos}")
    
    # Verificar archivos creados
    import os
    size_principal = os.path.getsize(archivo_principal)
    size_metadatos = os.path.getsize(archivo_metadatos)
    
    print(f"\n📁 Archivos creados:")
    print(f"   • {archivo_principal}: {size_principal:,} bytes")
    print(f"   • {archivo_metadatos}: {size_metadatos:,} bytes")
    
except Exception as e:
    print(f"❌ Error en exportación: {str(e)}")
    
# =============================================================================
# MOSTRAR MUESTRA DE RESULTADOS
# =============================================================================

print(f"\n🔍 Muestra de resultados (top 10 por densidad IDBA):")

columnas_muestra = ['cod_cuadrante', 'k_total', 'area_km2', 'density_idba_per_km2', 'shrinkage_factor']
columnas_muestra_disponibles = [col for col in columnas_muestra if col in df_export.columns]

muestra = df_export[columnas_muestra_disponibles].head(10)
print(muestra.to_string(index=False, float_format='%.4f'))

# =============================================================================
# RESUMEN FINAL
# =============================================================================

print(f"\n" + "="*80)
print("🎯 PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*80)
print(f"📊 Cuadrantes procesados: {len(df_export):,}")
print(f"📊 Muestras analizadas: {metadata['total_muestras']:,}")
print(f"📊 Área cubierta: {metadata['area_total_km2']:.2f} km²")
print(f"🧠 Metodología: {metadata['metodologia']}")
print(f"📅 Período: {metadata['periodo_datos']}")
print(f"🏙️ Ciudad: {metadata['ciudad']}")
print(f"💾 Archivos exportados en: tools/")
print("="*80)

💾 Preparando exportación de resultados...
📋 Columnas para exportar: 11/12
📊 Registros ordenados: 31

📈 Estadísticas finales:
   • fecha_procesamiento: 2025-10-12 23:04:52
   • total_cuadrantes: 31
   • cuadrantes_con_muestras: 31
   • cuadrantes_con_area: 30
   • total_muestras: 26977
   • area_total_km2: 13.6636
   • periodo_datos: 2021-2024
   • ciudad: MANIZALES
   • metodologia: IDBA Poisson-Gamma
   • densidad_idba_min: 66.1790
   • densidad_idba_max: 4955.6585
   • densidad_idba_media: 1949.0373
   • densidad_idba_mediana: 1345.3613
   • parametro_alpha: 0.9574
   • parametro_beta: 0.0005
   • metodo_parametros: MLE

💾 Exportando archivos...
   ✅ Tabla principal: indicadores_muestras_manizales_20251012_230452.csv
❌ Error en exportación: [Errno 2] No such file or directory: 'tools/metadatos_muestras_manizales_20251012_230452.json'

🔍 Muestra de resultados (top 10 por densidad IDBA):
cod_cuadrante  k_total  area_km2  density_idba_per_km2  shrinkage_factor
       MZ_012     3181    

: 

## 📊 Resultados y Conclusiones

### Resumen del Análisis

Este notebook implementa la metodología **IDBA (Empirical Bayes) con conjugada Poisson-Gamma** para estimar densidades de muestras por cuadrante en Manizales (2021-2024), proporcionando estimaciones más robustas que los enfoques naïve tradicionales.

### Metodología Aplicada

1. **Prior Conjugado**: λᵢ ~ Gamma(α, β)
2. **Likelihood**: kᵢ|λᵢ ~ Poisson(λᵢ × Aᵢ)  
3. **Posterior**: λᵢ|kᵢ ~ Gamma(α + kᵢ, β + Aᵢ)
4. **Estimador EB**: E[λᵢ|kᵢ] = (α + kᵢ)/(β + Aᵢ)

### Ventajas del Enfoque IDBA

- **Shrinkage hacia el prior**: Reduce variabilidad en cuadrantes pequeños
- **Intervalos de credibilidad**: Cuantifica incertidumbre
- **Robustez estadística**: Maneja mejor cuadrantes con pocas observaciones
- **Interpretabilidad**: Los parámetros α y β tienen significado epidemiológico

### Interpretación de Resultados

- **Densidades altas**: Cuadrantes con actividad intensiva de muestreo
- **Shrinkage factor**: Indica confiabilidad (valores altos = menos confiable)
- **Intervalos de credibilidad**: Reflejan incertidumbre en las estimaciones
- **Ranking**: Permite priorización basada en evidencia estadística

### Archivos Generados

- **CSV principal**: Tabla completa con densidades naïve e IDBA
- **Metadatos JSON**: Parámetros del modelo y estadísticas resumen
- **Rankings y percentiles**: Para análisis comparativo

### Próximos Pasos

1. Validar resultados con conocimiento local del terreno
2. Comparar con otros períodos temporales
3. Integrar con análisis espaciales (clustering, autocorrelación)
4. Desarrollar mapas interactivos para visualización